##### Consider the following input sentence, which has already been embedded into three-dimensional vectors

In [2]:
import torch
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your     (x^1)
     [0.55, 0.87, 0.66], # journey  (x^2)
     [0.57, 0.85, 0.64], # starts   (x^3)
     [0.22, 0.58, 0.33], # with     (x^4)
     [0.77, 0.25, 0.10], # one      (x^5)
     [0.05, 0.80, 0.55]  # step     (x^6)
    ]
)

The first step of implementing `self-ttention` is to compute the intermdiate values `w`, referred to as `attention scores`.

In [3]:
query = inputs[1] # The second input token serves as the query
attn_scores_2 = torch.empty(inputs.shape[0]) # Initialize an empty tensor to store attention scores

for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(query, x_i) # Compute the dot product between the query and each input token
    
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


Understanding `dot products`

A `dot product` is essentialy a concise way of multiplying two vectors element-wise and then summing the products, which can be demonstrated as follows :

In [4]:
res = 0
for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)

tensor(0.9544)


In [5]:
print(torch.dot(inputs[0], query))

tensor(0.9544)


The output confirms that sum of the element-wise multiplication gives the same results as the dot product.

---
Dans le contexte des mécanismes d’auto-attention, le `produit scalaire (dot product)` permet de déterminer dans quelle mesure un élément d’une séquence "porte attention" à un autre.
Plus le produit scalaire est grand, plus la similarité (et donc l’attention) entre deux éléments est importante.

---
In the next step, we `normalize` each of the **attention scores** we compute previously. The `main goal` of the normalization is `to obtain` **attention weights** `that sum up to 1`. This normalization is a convention that `is useful for interpretation and maintaining training stability` in an LLM.

In [6]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In practice, `it's more common and advisable to use` **softmax function** `for normalization`. This approach `is better at managing extreme values and offers more favorable gradient properties` during training.

In [7]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention_weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention_weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In addition, the `softmax function ensures that the attention weights are always positive`. This makes the output **interpretable as probabilities or relative importance**, where **higher weights indicate greater importance**.


Note that this naive softmax implementation (`softmax_naive`) may `encounter numerical instability problems` such as **overflow** and **underflow**, when dealing with large or small input value.

Therefore in practice, it's advisable to `use the Pytorch implementation of sofmax`, which has been extensively optimized for performance.

In [8]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention_weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention_weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


Npw that we have computed the normalized attention weights, we are going to calculate the **context vector** $z^{(2)}$ by **multiplying the embeddeed input token $x(i)$**, with **the corresponding attention weights and then summing the vectors**.

In [9]:
query = inputs[1] # The second input token is the query
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


##### Computing attention weights for all inputs tokens

In [10]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


When computing the preceding attention score tensor, we use `for` loops in Python.

However, `for` loops are generally **slow**, and we can achieve the same results using matrix multiplication.

In [11]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In step `2`, we normalize each row so that **the values in each row sum to 1**.

In [12]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In the context of using Pytorch, the `dim` parameter in function like `torch.softmax` specifies **the dimension of the input tensor along which the function will be computed**.

By setting `dim=-1`, we are instructing `softmax` function to apply the normalization along **the last dimension** of the `attn_scores` tensor.
If `attn_scores` is a two-dimensional tensor (for example, wita shape of **[rows, columns]**), it will **normalize across the colunms** so that the values in each row sum up to 1.

---
In the third and final step, we use the attention weights to compute all **context vectors** via matrix multiplication.

In [13]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


We can double-check that the code is correct by compaining the second row with the context vector $z^{(2)}$ that we computed early.

In [14]:
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


This concludes the code walkthrough of a simple **self-attention mechanism**.

#### Inplementing self-attention with trainable weights

##### - Computing the attention weights step by step

In [15]:
x_2 = inputs[1] # The second input element
d_in = inputs.shape[1] # The input embedding size, d = 3
d_out = 2 # The output embedding size d_out = 2

**Note that in GPT-like models, the input and output dimensions are usually the same.**

Next, we initialize the three weight matrices $W_{q}$, $W_{k}$ et $W_{v}$.

In [26]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

Next we compute the **query, key and value vectors**.

In [27]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print("Query vector for x_2:", query_2)

Query vector for x_2: tensor([0.4306, 1.4551])


Dans les matrices de poids **$W$**, le terme « poids » fait référence à des **paramètres entraînables**, c’est-à-dire aux valeurs d’un réseau de neurones optimisées pendant l’apprentissage.
Il ne faut pas les confondre avec les **poids d’attention**.

Comme nous l’avons déjà vu, les poids d’attention déterminent dans quelle mesure un vecteur de contexte dépend des différentes parties de l’entrée
(autrement dit, sur quelles parties de l’entrée le réseau se concentre).

We can obtain **key** and **value** vectors for all input elements via matrix multiplication.

In [28]:
keys = inputs @ W_key
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


As we can tell from the outputs, we sucessfully projected the six input tokens from a three-dimensional onto a two-dimensional embedding space.

---

The second step is to compute the attention scores.
First, let's compute the **attention score $w_{22}$**

In [29]:
keys_2 = keys[1] # The key vector for the second input token
attn_score_22 = query_2 @ keys_2 # Compute the attention score between the query and the key of the second token
print("Attention score between query_2 and key_2:", attn_score_22)

Attention score between query_2 and key_2: tensor(1.8524)


We can generalize this computation to all attention scores via matrix multiplication.

In [30]:
print("query_2.shape:", query_2.shape)
print("keys.shape:", keys.shape)

query_2.shape: torch.Size([2])
keys.shape: torch.Size([6, 2])


In [31]:
attn_scores_2 = query_2 @ keys.T # All attention scores for given query
print("Attention scores for query_2:", attn_scores_2)

Attention scores for query_2: tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


Now, we want to go from the attention scores to the **attention weights** by **scaling attention scores and using the softmax function**.

However, we **scale the attention scores by dividing them by the square root of the embedding dimension of keys**

In [32]:
d_k = keys.shape[1] # The embedding dimension of the keys
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print("Attention weights for query_2:", attn_weights_2)

Attention weights for query_2: tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


Nous allons maintenant calculer le **vecteur de contexte** comme somme pondérée des vecteurs de valeur.
Les poids d’attention servent ici à pondérer l’importance relative de chaque vecteur.

In [33]:
context_vec_2 = attn_weights_2 @ values
print("Context vector for query_2:", context_vec_2)

Context vector for query_2: tensor([0.3061, 0.8210])


##### - Implementing a compact self-attention Python class

A compact self-attention class

In [ ]:
import torch.nn as nn
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys    = x @ self.W_key
        queries = x @ self.W_query
        values  = x @ self.W_value

        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vecs = attn_weights @ values

        return context_vecs

We can use this class as follow.

In [37]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


As a quick check, notice that the second row **$([0.3061, 0.8210])$** matches the contents of `context_vec_2` in the previous section.

##### A self-attention class using Pytorch's Linear layers

In [43]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)

        attn_scores  = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vecs = attn_weights @ values

        return context_vecs

In [44]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


##### Exercice 3.1

In [63]:
sa_v1.W_key   = torch.nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_query = torch.nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_value = torch.nn.Parameter(sa_v2.W_value.weight.T)

print(sa_v1(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
